Can be run as script:
```bash
FLAT_PKL_FILE="results/rmappo-MultiAgentFishEnv-2MNoAttn/20250708_093013/outputs/MAFish_neural_20250708_093013_EEeQBDAM_agg_flattened.pkl"
jupyter nbconvert report_behavior_predict.ipynb --to python 
python -u report_behavior_predict.py --flat_pkl_file $FLAT_PKL_FILE

# Extra options
python -u report_behavior_predict.py --flat_pkl_file $FLAT_PKL_FILE --model_seed 1 --name_mode name
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import argparse
import os
import glob
import tqdm

import utils_report as ru
from utils_behavior import (
    bin_agent_size,
    plot_behavior_densities_1d,
)
from utils_sensors import get_sensor_indices_from_cfg

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import rcParams
import seaborn as sns

rcParams["pdf.fonttype"] = 42  # Use Type 42 (TrueType) fonts to save text as text
rcParams["ps.fonttype"] = 42  # For PostScript as well, if needed


import utils_decoding as du
from cfg import FEATURE_METADATA, EXCLUDE_FROM_DECODING

from utils_predict import (
    setup_data,
    assign_discrete_behavior_labels,
    assign_discrete_behavior_labels_equal_mass,
    shift_and_train_test_split_data,
    train_assess_classifier,
)
from scipy.stats import reciprocal, uniform, randint

# Check if we're in interactive mode or batch mode
batchmode = False
if "ipykernel_launcher" in sys.argv[0]:
    print("Interactive mode")
else:
    batchmode = True
    print("Batch/CLI mode")
    # Parses the command line arguments below


## Load data

In [ ]:
TASK= "foraging"  # Default task
BASE_PATH = "./"
# SUB_PATH="results/rmappo-MultiAgentFishEnv-2MAttnNoShaping/20250708_093251/outputs/MAFish_neural_20250708_093251_Iw492bxU_agg_flattened.pkl"
SUB_PATH = "results/rmappo-MultiAgentFishEnv-2MNoAttn/20250708_093013/outputs/MAFish_neural_20250708_093013_EEeQBDAM_agg_flattened.pkl"
# SUB_PATH="results/rmappo-MultiAgentFishEnv-2MAttn/20250708_092959/outputs/MAFish_neural_20250708_092959_MOsJFQ60_agg_flattened.pkl"
flat_pkl_file = f"{BASE_PATH}/{SUB_PATH}"

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--flat_pkl_file", type=str, help="Path to input pickle file")
parser.add_argument(
    "--model_seed", type=int, default=137, help="Random seed for classifier"
)
parser.add_argument(
    "--name_mode",
    type=str,
    default="name",
    choices=["name", "short_name"],
    help="Feature label mode",
)
parser.add_argument(
    "--task",
    type=str,
    default="foraging",
    help="Task e.g., 'foraging', '2fip', etc.",
)
args = parser.parse_args()

if batchmode:
    flat_pkl_file = args.flat_pkl_file
    model_seed = args.model_seed
    task = args.task
    name_mode = args.name_mode
else:
    # fallback for interactive mode
    flat_pkl_file = flat_pkl_file
    model_seed = args.model_seed
    task = args.task
    name_mode = args.name_mode


In [ ]:
dff, OUTPUTS_FOLDER, pkl_str = ru.load_flat_pkl_file(flat_pkl_file, task=task)

# Behavior and observations

In [ ]:
# try:
#     m_idx, a_idx, k_idx = ru.get_sensor_indices(dff)
#     print(m_idx, a_idx, k_idx)
# except Exception as e:
#     try:
#         m_idx, a_idx, k_idx = get_sensor_indices_from_cfg()
#         m_idx = slice(m_idx[0], a_idx[0])  # Mormyromast sensors
#         a_idx = slice(a_idx[0], k_idx[0])  # Ampullary sensors
#         k_idx = slice(k_idx[0], k_idx[-1] + 1)  # Knollen sensors
#         print(m_idx, a_idx, k_idx)
#     except Exception as e2:
#         print(
#             "Error getting sensor indices from data or config. Please check the data format."
#         )
#         print("Error message:", e)
#         print("Error message from config:", e2)
#         sys.exit(1)


# observation_width = dff["observations"].values[0].shape[0]
# sensor_types = [
#     ("Mormyromast", m_idx),
#     ("Ampullary", a_idx),
#     ("Knollen", k_idx),
#     ("Other", slice(k_idx.stop, observation_width)),
# ]

In [ ]:
# dff, bin_labels = bin_agent_size(dff, num_bins=3, bin_column="size_bin")
# plot_behavior_densities_1d(dff)

# Behavior decoder

## Feature extraction/check

In [ ]:
dff, INDEPENDENTS = setup_data(dff, behavior_mode="foraging", FEATURE_METADATA=FEATURE_METADATA, EXCLUDE_FROM_DECODING=EXCLUDE_FROM_DECODING,)

## Behavior prediction

In [ ]:
binning_method = ["equal_mass", "predefined"][0]  # Options
model_seed = 137
batchmode = False
HYPOPT_ITER = 20
HYPOPT_CV = 3
if batchmode:
    HYPOPT_ITER = 20
    HYPOPT_CV = 3

for action_var in ["moveturn", "eod"]:
    print(f"Processing action variable: {action_var}")
    if binning_method == "predefined":
        dff = assign_discrete_behavior_labels(dff)
    if binning_method == "equal_mass":
        dff = assign_discrete_behavior_labels_equal_mass(dff, action_var=action_var)

    X_train, y_train, X_test, y_test, INDEPENDENTS = shift_and_train_test_split_data(dff, INDEPENDENTS)

    classifier_df = train_assess_classifier(
        X_train, y_train, 
        X_test, y_test,
        action_var=action_var,
        param_distributions = {"n_estimators": randint(10, 100), "max_depth": randint(2, 20),},
        FEATURE_METADATA=FEATURE_METADATA,
        model_seed=137,
        HYPOPT_ITER=20,
        HYPOPT_CV=3,
        verbose_rf=2,
        n_jobs_rf=20,
        name_mode=args.name_mode,
        OUTPUTS_FOLDER=OUTPUTS_FOLDER,
    )
    print(f"Classifier results saved to {OUTPUTS_FOLDER}")